In [1]:
# Установка необходимых библиотек
!pip install -q opendatasets lmdb fire natsort

import os
import opendatasets as od
# Скачиваем данные
od.download("https://www.kaggle.com/datasets/ziiiegen/price-ocr-data")

#Клонируем FORK репозиторий ClovaAI
!git clone https://github.com/ziiiegen/deep-text-recognition-benchmark.git
import sys
sys.path.append('/content/deep-text-recognition-benchmark')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.9 MB/s eta 0:00:00
Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: ziiiegen
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/ziiiegen/price-ocr-data


100%|██████████| 23.5M/23.5M [00:02<00:00, 9.88MB/s]



Cloning into 'deep-text-recognition-benchmark'...
remote: Enumerating objects: 545, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 545 (delta 239), reused 234 (delta 218), pack-reused 274 (from 1)
Receiving objects: 100% (545/545), 3.07 MiB | 27.80 MiB/s, done.
Resolving deltas: 100% (339/339), done.


In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image

# Импорты из библиотеки ClovaAI
from dataset import AlignCollate
from model import Model
from utils import CTCLabelConverter

# Пути
TEST_IMG_DIR = "/content/price-ocr-data/OCR_DATA/test/test"
SUBMISSION_PATH = "/content/price-ocr-data/OCR_DATA/sample_submission.csv"
OUTPUT_PATH = "/content/submission.csv"

MODEL_PATH_1 = "/content/CTC_first.pth"
MODEL_PATH_2 = "/content/CTC_fine_tuning.pth"

WEIGHT_1 = 0.97
WEIGHT_2 = 0.98

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Конфиг
class Opt:
    batch_max_length = 6
    imgH = 96
    imgW = 110
    rgb = False
    character = "0123456789"
    sensitive = False
    PAD = True

    Transformation = "TPS"
    FeatureExtraction = "ResNet"
    SequenceModeling = "BiLSTM"
    Prediction = "CTC"

    num_fiducial = 4
    input_channel = 1
    output_channel = 512
    hidden_size = 256

opt = Opt()

# Инициализация
converter = CTCLabelConverter(opt.character)
opt.num_class = len(converter.character)

def load_model(path, opt, device):
    """Загрузка весов в архитектуру ClovaAI"""
    print(f"Loading model from: {path}")
    model = Model(opt)
    state_dict = torch.load(path, map_location=device)
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model

# Загружаем обе модели
model_1 = load_model(MODEL_PATH_1, opt, device)
model_2 = load_model(MODEL_PATH_2, opt, device)

# Подготовка изображений (ресайз, паддинг)
align_collate = AlignCollate(
    imgH=opt.imgH,
    imgW=opt.imgW,
    keep_ratio_with_pad=opt.PAD
)

def get_prediction_and_confidence(model, image_tensor):
    """
    Прогоняет тензор через модель и возвращает:
    1. Очищенную строку (только цифры)
    2. Уверенность модели в предсказании (0.0 - 1.0)
    """
    with torch.no_grad():
        preds = model(image_tensor, text=None) # [batch, T, num_class]

        # Переводим логиты в вероятности [0..1]
        probs = torch.softmax(preds, dim=2)

        # Находим максимальную вероятность для каждого "кадра"
        max_probs, preds_index = probs.max(2)

        # Декодируем (убираем дубликаты и blank токены)
        preds_size = torch.IntTensor([preds.size(1)])
        pred_str = converter.decode(preds_index, preds_size)[0]

        # --- Расчет Confidence (уверенности) ---
        # В CTC индекс 0 зарезервирован под [blank] токен.
        # Мы считаем уверенность только по значимым символам (не blank).
        blank_index = 0
        non_blank_mask = preds_index[0] != blank_index

        if non_blank_mask.sum() > 0:
            # Средняя вероятность по всем непустым предсказанным кадрам
            confidence = max_probs[0][non_blank_mask].mean().item()
        else:
            # Если модель предсказала только пустоту
            confidence = 0.0

        # Очистка: оставляем только цифры, на случай если вылезет мусор
        pred_str_clean = "".join(ch for ch in pred_str if ch.isdigit())

    return pred_str_clean, confidence


def predict_ensemble(img_path):
    """Ансамбль: сравнение результатов двух моделей"""
    if not os.path.exists(img_path):
        return ""

    try:
        image = Image.open(img_path).convert("L")
    except Exception:
        return ""

    # Готовим тензор один раз для обеих моделей
    image_tensor, _ = align_collate([(image, "")])
    image_tensor = image_tensor.to(device)

    # Получаем результаты
    pred_1, conf_1 = get_prediction_and_confidence(model_1, image_tensor)
    pred_2, conf_2 = get_prediction_and_confidence(model_2, image_tensor)

    # Применяем веса к уверенности (по умолчанию не меняет ничего, т.к. вес = 1.0)
    weighted_conf_1 = conf_1 * WEIGHT_1
    weighted_conf_2 = conf_2 * WEIGHT_2

    # --- Логика выбора (Late Fusion) ---
    if pred_1 == pred_2:
        return pred_1  # Модели согласны друг с другом
    else:
        # Модели спорят. Выбираем ту, чья взвешенная уверенность выше
        if weighted_conf_1 >= weighted_conf_2:
            return pred_1
        else:
            return pred_2

# Запуск инференса на тестовом датасете
if __name__ == '__main__':
    print("\n--- Starting Ensemble Test Prediction ---")

    submission = pd.read_csv(SUBMISSION_PATH)
    predictions = []

    # Проходим по всем картинкам с прогресс-баром
    for image_id in tqdm(submission["Filename"], desc="Predicting"):
        img_path = os.path.join(TEST_IMG_DIR, image_id)

        # Получаем финальный ответ от ансамбля
        final_pred = predict_ensemble(img_path)
        predictions.append(final_pred)

    # Сохраняем в CSV
    submission["Price"] = predictions
    submission.to_csv(OUTPUT_PATH, index=False)

    print(f"\nDone! Submission saved to: {OUTPUT_PATH}")

Loading model from: /content/11_ctc.pth
Loading model from: /content/14_ctc.pth

--- Starting Ensemble Test Prediction ---


Predicting: 100%|██████████| 3762/3762 [01:49<00:00, 34.51it/s]


Done! Submission saved to: /content/submissionW2.csv
